# PyOD Quickstart — Unsupervised Outlier Detection

This notebook walks through a minimal PyOD workflow (2019):

1. Generate a 2D synthetic dataset with known outliers
2. Fit a kNN detector
3. Inspect labels / scores and a simple metric

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from pyod.models.knn import KNN
from pyod.utils.data import generate_data
from sklearn.metrics import roc_auc_score

## 1. Synthetic data

`contamination` is the expected outlier ratio used both for data generation and for thresholding scores.

In [ ]:
contamination = 0.15
X_train, y_train = generate_data(
    n_train=400,
    n_features=2,
    contamination=contamination,
    train_only=True,
    random_state=42,
)

plt.figure(figsize=(5, 4))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='steelblue', s=18, label='inlier')
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='crimson', s=22, label='outlier')
plt.legend()
plt.title('Ground truth')
plt.show()
print('n =', len(y_train), '| outliers =', int(y_train.sum()))

## 2. Fit kNN detector

Anomaly score ≈ distance to the k-th neighbor (larger ⇒ more anomalous).

In [ ]:
clf = KNN(contamination=contamination, n_neighbors=10)
clf.fit(X_train)

y_pred = clf.labels_          # 0/1 after thresholding by contamination
y_scores = clf.decision_scores_

auc = roc_auc_score(y_train, y_scores)
print('ROC-AUC = {:.4f}'.format(auc))
print('flagged outliers =', int(y_pred.sum()))

## 3. Visualize anomaly scores

In [ ]:
plt.figure(figsize=(5, 4))
sc = plt.scatter(X_train[:, 0], X_train[:, 1], c=y_scores, cmap='RdBu_r', s=22)
plt.colorbar(sc, label='anomaly score')
plt.title('kNN anomaly scores')
plt.show()

## Next steps

- Open `02_model_comparison.ipynb` for multi-model ranking
- Run `python benchmark_odds.py` for ODDS real-data results